# 02. Subcellular Model Benchmarking & Ablations

## Overview
This notebook evaluates model architectures, pooling strategies, and loss formulations across the 11-compartment subcellular localization benchmark.

### Models Evaluated:
1. **Random Control (`ctrl`)**: Shuffled-label baseline.
2. **Logistic Regression (`LR`)**: Linear classifier on 1280-dim ESM embeddings.
3. **1-Hidden MLP (`M1`)**: $1280 \to 64 \to \text{ReLU} \to 11$.
4. **2-Hidden MLP (`M2`)**: $1280 \to 128 \to \text{ReLU} \to 64 \to \text{ReLU} \to 11$.

### Ablations:
- **Loss Variants**: Standard BCE, Class-weighted BCE ($\ddagger$), Focal Loss ($\bigstar$).
- **Regularization**: Ridge Frobenius penalty $\lambda=10^{-4}$ ($\dagger$).
- **Pooling**: Mean vs. Max sequence pooling.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import protein_loc as pl

%matplotlib inline
pl.set_seed(42)

## 1. Load Data

In [ ]:
train_df_mean, X_mean, y_train = pl.load_subcellular_data('../data/df_train_loc_mean.csv')
train_df_max, X_max, _ = pl.load_subcellular_data('../data/df_train_loc_max.csv')

print(f"Loaded {len(train_df_mean):,} samples with partitions: {train_df_mean['Partition'].unique().tolist()}")

## 2. Run Cross-Validation Across Models & Formulations

We evaluate each model using leave-one-homology-partition-out cross-validation.

In [ ]:
results = {}

# 1. Random Control
print("Evaluating Random Control...")
y_shuffled = np.random.permutation(y_train)
df_shuffled = train_df_mean.copy()
for i, loc in enumerate(pl.SUBCELLULAR_LOCATIONS):
    df_shuffled[loc] = y_shuffled[:, i]
results['ctrl'] = pl.run_cross_validation(df_shuffled, lambda: pl.LogisticRegression(), epochs=10)

# 2. Logistic Regression (Standard BCE)
print("Evaluating Logistic Regression (LR)...")
results['LR'] = pl.run_cross_validation(train_df_mean, lambda: pl.LogisticRegression(), epochs=100)

# 3. Logistic Regression (Class-Weighted BCE ‡)
print("Evaluating Logistic Regression (LR‡)...")
results['LR‡'] = pl.run_cross_validation(
    train_df_mean, lambda: pl.LogisticRegression(), epochs=100,
    criterion_factory=lambda w: pl.weighted_bce(w)
)

# 4. Logistic Regression (Focal Loss ★)
print("Evaluating Logistic Regression (LR★)...")
results['LR★'] = pl.run_cross_validation(
    train_df_mean, lambda: pl.LogisticRegression(), epochs=100,
    criterion_factory=lambda _: pl.focal_loss
)

# 5. 1-Hidden MLP (M1)
print("Evaluating 1-Hidden MLP (M1)...")
results['M1'] = pl.run_cross_validation(train_df_mean, lambda: pl.get_model('mlp_1h'), epochs=100)

# 6. 1-Hidden MLP (Class-Weighted M1‡)
print("Evaluating 1-Hidden MLP (M1‡)...")
results['M1‡'] = pl.run_cross_validation(
    train_df_mean, lambda: pl.get_model('mlp_1h'), epochs=100,
    criterion_factory=lambda w: pl.weighted_bce(w)
)

# 7. 1-Hidden MLP (Focal Loss M1★)
print("Evaluating 1-Hidden MLP (M1★)...")
results['M1★'] = pl.run_cross_validation(
    train_df_mean, lambda: pl.get_model('mlp_1h'), epochs=100,
    criterion_factory=lambda _: pl.focal_loss
)

# 8. 1-Hidden MLP + Ridge Regularization (M1†)
print("Evaluating 1-Hidden MLP + Ridge (M1†)...")
results['M1†'] = pl.run_cross_validation(
    train_df_mean, lambda: pl.get_model('mlp_1h'), epochs=100,
    regularizer=pl.ridge_penalty(1e-4)
)

# 9. 2-Hidden MLP (M2)
print("Evaluating 2-Hidden MLP (M2)...")
results['M2'] = pl.run_cross_validation(train_df_mean, lambda: pl.get_model('mlp_2h'), epochs=100)

# 10. 2-Hidden MLP + Ridge Regularization (M2†)
print("Evaluating 2-Hidden MLP + Ridge (M2†)...")
results['M2†'] = pl.run_cross_validation(
    train_df_mean, lambda: pl.get_model('mlp_2h'), epochs=100,
    regularizer=pl.ridge_penalty(1e-4)
)

print("\nCross-validation benchmarks completed successfully!")

## 3. Comparative Benchmark Summary Table

In [ ]:
summary_rows = []
for name, res in results.items():
    m = res['mean']
    s = res['std']
    summary_rows.append({
        'Model': name,
        'Exact Match': f"{m['exact_match']*100:.1f} ± {s['exact_match']*100:.1f}%",
        'Jaccard': f"{m['jaccard']:.3f} ± {s['jaccard']:.3f}",
        'Micro-F1': f"{m['micro_f1']:.3f} ± {s['micro_f1']:.3f}",
        'Macro-F1': f"{m['macro_f1']:.3f} ± {s['macro_f1']:.3f}",
        'Mean MCC': f"{m['mean_mcc']:.3f} ± {s['mean_mcc']:.3f}",
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

## 4. Per-Class MCC Heatmap Matrix

We visualize the cross-validated per-compartment MCC across all models.

In [ ]:
model_order = ['ctrl', 'LR', 'LR‡', 'LR★', 'M1', 'M1‡', 'M1★', 'M1†', 'M2', 'M2†']
mcc_matrix = np.zeros((len(model_order), len(pl.SUBCELLULAR_LOCATIONS)))

for i, m_name in enumerate(model_order):
    for j, loc in enumerate(pl.SUBCELLULAR_LOCATIONS):
        mcc_matrix[i, j] = results[m_name]['per_class_mcc_mean'][loc]

fig, ax = pl.plot_cross_validated_mcc_heatmap(
    mcc_data=mcc_matrix,
    model_names=model_order,
    locations=pl.SUBCELLULAR_LOCATIONS,
    save_path='../figures/cross_validated_mcc_heatmap.pdf'
)
plt.show()